# Linear Time Sorting Algorithms

This notebook covers **non-comparison-based** sorting algorithms that can achieve **O(n)** time complexity under certain conditions.

## Table of Contents
1. [Why Can Linear Sorts Beat O(n log n)?](#why-linear)
2. [Counting Sort](#counting-sort)
3. [Radix Sort](#radix-sort)
4. [Bucket Sort](#bucket-sort)
5. [Complexity Comparison](#complexity-comparison)
6. [Key Takeaways](#key-takeaways)

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook builds algorithmic thinking used to reason about performance and correctness in computational biology tools.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Big-O is asymptotic guidance; constant factors still matter in practice.
- Correctness comes before optimization: verify edge cases before performance tuning.
- Data structure choice often dominates speed more than micro-optimizations.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Comparison-Based Sorting Algorithms](../../Tier_4_Algorithms_and_Data_Structures/02_Sorting_Algorithms/01_comparison_sorts.ipynb) | [Next: Linear and Binary Search Algorithms →](../../Tier_4_Algorithms_and_Data_Structures/03_Searching_Algorithms/01_linear_binary_search.ipynb)

---

---

<a id="why-linear"></a>
## Why Can These Algorithms Beat O(n log n)?

### The Comparison Sort Lower Bound

**Theorem:** Any comparison-based sorting algorithm requires **Ω(n log n)** comparisons in the worst case.

**Proof Intuition:**
- With `n` elements, there are `n!` possible permutations
- Each comparison gives 1 bit of information (≤ or >)
- To distinguish `n!` orderings, we need at least `log₂(n!)` comparisons
- By Stirling's approximation: `log₂(n!) ≈ n log₂(n)`

### How Linear Sorts Escape This Bound

**They don't compare elements!** Instead, they use:
- **Direct addressing** (Counting Sort): Elements map directly to positions
- **Digit decomposition** (Radix Sort): Sort by individual digits
- **Distribution** (Bucket Sort): Elements distributed to buckets by value

### The Trade-off

| Advantage | Cost |
|-----------|------|
| O(n) time | Requires assumptions about input |
| Stable by design | Extra space for auxiliary structures |
| Predictable performance | Limited to specific data types |

---

<a id="counting-sort"></a>
## 1. Counting Sort

### Theory

**Counting Sort** sorts integers by counting occurrences of each value, then using arithmetic to place elements in their correct positions.

#### Assumptions & Limitations
- Input must be **integers** (or keys that map to integers)
- Range of values `k` must be known and **reasonable** (O(n) or close)
- If `k >> n`, space and time become O(k), making it inefficient

#### When to Use
- Sorting integers with a **small, known range** (e.g., ages 0-120, grades 0-100)
- When **stability** is required
- As a subroutine in **Radix Sort**

### ASCII Art Visualization

```
Input: [4, 2, 2, 8, 3, 3, 1]
Range: 1-8

Step 1: Count occurrences
┌─────────────────────────────────────────┐
│ Index: 0   1   2   3   4   5   6   7   8│
│ Count: 0   1   2   2   1   0   0   0   1│
│            ↑   ↑   ↑   ↑               ↑│
│           (1) (2) (3) (4)             (8)│
└─────────────────────────────────────────┘

Step 2: Cumulative sum (prefix sum)
┌─────────────────────────────────────────┐
│ Index: 0   1   2   3   4   5   6   7   8│
│ Count: 0   1   3   5   6   6   6   6   7│
│            │   │   │   │               │ │
│            ▼   ▼   ▼   ▼               ▼ │
│     "1 goes" "2s go" "3s go" "4 goes" "8 goes"│
│     to pos 0 to 1-2 to 3-4  to pos 5  to pos 6│
└─────────────────────────────────────────┘

Step 3: Build output (right to left for stability)
┌─────────────────────────────────────────┐
│ Processing: 1 ← 3 ← 3 ← 8 ← 2 ← 2 ← 4  │
│                                         │
│ Output: [1, 2, 2, 3, 3, 4, 8]           │
│          ↑  ↑  ↑  ↑  ↑  ↑  ↑            │
│         pos 0  1  2  3  4  5  6         │
└─────────────────────────────────────────┘
```

In [ ]:
from typing import List, Optional


def counting_sort(arr: List[int], max_val: Optional[int] = None) -> List[int]:
    """
    Sort an array of non-negative integers using Counting Sort.
    
    This is a stable, non-comparison-based sorting algorithm that works
    by counting the occurrences of each unique value.
    
    Args:
        arr: List of non-negative integers to sort
        max_val: Maximum value in array (computed if not provided)
    
    Returns:
        New sorted list
    
    Time Complexity: O(n + k) where k is the range of values
    Space Complexity: O(k) for the count array + O(n) for output
    
    Example:
        >>> counting_sort([4, 2, 2, 8, 3, 3, 1])
        [1, 2, 2, 3, 3, 4, 8]
    """
    if not arr:
        return []
    
    # Determine the range
    k = max_val if max_val is not None else max(arr)
    
    # Step 1: Count occurrences of each value
    count = [0] * (k + 1)
    for num in arr:
        count[num] += 1
    
    # Step 2: Compute cumulative counts (prefix sum)
    # count[i] now contains the number of elements <= i
    for i in range(1, k + 1):
        count[i] += count[i - 1]
    
    # Step 3: Build output array (iterate right-to-left for stability)
    output = [0] * len(arr)
    for num in reversed(arr):
        # count[num] - 1 gives the correct position for num
        count[num] -= 1
        output[count[num]] = num
    
    return output

In [ ]:
def counting_sort_verbose(arr: List[int]) -> List[int]:
    """
    Counting Sort with step-by-step visualization.
    
    Args:
        arr: List of non-negative integers to sort
    
    Returns:
        Sorted list
    """
    if not arr:
        return []
    
    print(f"Input array: {arr}")
    print(f"Length: {len(arr)}")
    print("=" * 60)
    
    k = max(arr)
    print(f"\nStep 1: Create count array (size = max + 1 = {k + 1})")
    
    # Count occurrences
    count = [0] * (k + 1)
    for i, num in enumerate(arr):
        count[num] += 1
        print(f"  arr[{i}] = {num} → count[{num}]++")
    
    print(f"\n  Count array: {count}")
    print(f"  Indices:     {list(range(len(count)))}")
    
    # Cumulative sum
    print("\nStep 2: Compute cumulative sum (prefix sum)")
    for i in range(1, k + 1):
        old_val = count[i]
        count[i] += count[i - 1]
        print(f"  count[{i}] = {old_val} + count[{i-1}] = {count[i]}")
    
    print(f"\n  Cumulative:  {count}")
    print(f"  Meaning: count[v] = number of elements ≤ v")
    
    # Build output
    print("\nStep 3: Build output (right to left for stability)")
    output = [0] * len(arr)
    
    for num in reversed(arr):
        pos = count[num] - 1
        count[num] -= 1
        output[pos] = num
        print(f"  Place {num} at position {pos} → output = {output}")
    
    print("\n" + "=" * 60)
    print(f"Sorted array: {output}")
    
    return output

In [ ]:
# Example: Step-by-step execution
print("COUNTING SORT DEMONSTRATION")
print("=" * 60)

test_array = [4, 2, 2, 8, 3, 3, 1]
result = counting_sort_verbose(test_array)

In [ ]:
# Test stability: elements with same key maintain relative order
print("\nSTABILITY TEST")
print("=" * 60)

# Using tuples (value, original_index) to track stability
def counting_sort_with_objects(arr: List[tuple]) -> List[tuple]:
    """Sort tuples by first element (key), demonstrating stability."""
    if not arr:
        return []
    
    k = max(item[0] for item in arr)
    count = [0] * (k + 1)
    
    for item in arr:
        count[item[0]] += 1
    
    for i in range(1, k + 1):
        count[i] += count[i - 1]
    
    output = [None] * len(arr)
    for item in reversed(arr):  # Right-to-left for stability!
        count[item[0]] -= 1
        output[count[item[0]]] = item
    
    return output

# Items: (key, identifier)
items = [(3, 'A'), (1, 'B'), (3, 'C'), (2, 'D'), (3, 'E')]
print(f"Input:  {items}")
print("        Note: Three items with key=3: A, C, E")

sorted_items = counting_sort_with_objects(items)
print(f"Output: {sorted_items}")
print("        Items with key=3 maintain order: A, C, E ✓")

---

<a id="radix-sort"></a>
## 2. Radix Sort

### Theory

**Radix Sort** sorts numbers by processing individual digits, from least significant to most significant (LSD) or vice versa (MSD).

#### Key Insight
If we use a **stable** sort for each digit position, the overall sort is correct.

#### Assumptions & Limitations
- Works on integers or fixed-length strings
- All numbers must have the same number of digits (pad with leading zeros)
- Requires a stable sorting algorithm for each digit (typically Counting Sort)

#### When to Use
- Sorting **large numbers** with many digits (where comparison would be expensive)
- Sorting **strings** of fixed length
- When `d × (n + k) < n log n` (d = digits, k = base/radix)

### ASCII Art Visualization

```
Input: [170, 45, 75, 90, 802, 24, 2, 66]

Pad to 3 digits: [170, 045, 075, 090, 802, 024, 002, 066]

═══════════════════════════════════════════════════════════════
Pass 1: Sort by 1s digit (rightmost)
═══════════════════════════════════════════════════════════════

  170  045  075  090  802  024  002  066
    ↓    ↓    ↓    ↓    ↓    ↓    ↓    ↓
    0    5    5    0    2    4    2    6

  Buckets:  0: [170, 090]
            2: [802, 002]
            4: [024]
            5: [045, 075]
            6: [066]

  Result: [170, 090, 802, 002, 024, 045, 075, 066]

═══════════════════════════════════════════════════════════════
Pass 2: Sort by 10s digit (middle)
═══════════════════════════════════════════════════════════════

  170  090  802  002  024  045  075  066
   ↓    ↓    ↓    ↓    ↓    ↓    ↓    ↓
   7    9    0    0    2    4    7    6

  Buckets:  0: [802, 002]
            2: [024]
            4: [045]
            6: [066]
            7: [170, 075]
            9: [090]

  Result: [802, 002, 024, 045, 066, 170, 075, 090]

═══════════════════════════════════════════════════════════════
Pass 3: Sort by 100s digit (leftmost)
═══════════════════════════════════════════════════════════════

  802  002  024  045  066  170  075  090
  ↓    ↓    ↓    ↓    ↓    ↓    ↓    ↓
  8    0    0    0    0    1    0    0

  Buckets:  0: [002, 024, 045, 066, 075, 090]
            1: [170]
            8: [802]

  Result: [002, 024, 045, 066, 075, 090, 170, 802]

═══════════════════════════════════════════════════════════════
Final: [2, 24, 45, 66, 75, 90, 170, 802] ✓
═══════════════════════════════════════════════════════════════
```

In [ ]:
def counting_sort_by_digit(arr: List[int], exp: int) -> List[int]:
    """
    Helper function: Sort array by a specific digit position using Counting Sort.
    
    Args:
        arr: List of non-negative integers
        exp: The digit position (1 for units, 10 for tens, 100 for hundreds, etc.)
    
    Returns:
        Array sorted by the specified digit
    
    Note:
        Uses base 10, so digit values are 0-9.
    """
    n = len(arr)
    output = [0] * n
    count = [0] * 10  # Digits 0-9
    
    # Count occurrences of each digit at position 'exp'
    for num in arr:
        digit = (num // exp) % 10
        count[digit] += 1
    
    # Cumulative count
    for i in range(1, 10):
        count[i] += count[i - 1]
    
    # Build output (right-to-left for stability)
    for num in reversed(arr):
        digit = (num // exp) % 10
        count[digit] -= 1
        output[count[digit]] = num
    
    return output


def radix_sort(arr: List[int]) -> List[int]:
    """
    Sort an array of non-negative integers using Radix Sort (LSD).
    
    Uses Counting Sort as the stable subroutine for each digit.
    Processes digits from least significant to most significant.
    
    Args:
        arr: List of non-negative integers to sort
    
    Returns:
        New sorted list
    
    Time Complexity: O(d × (n + k)) where:
        - d = number of digits in the maximum number
        - n = number of elements
        - k = base (10 for decimal)
    
    Space Complexity: O(n + k)
    
    Example:
        >>> radix_sort([170, 45, 75, 90, 802, 24, 2, 66])
        [2, 24, 45, 66, 75, 90, 170, 802]
    """
    if not arr:
        return []
    
    # Find the maximum number to determine the number of digits
    max_num = max(arr)
    
    # Make a copy to avoid modifying the original
    result = arr.copy()
    
    # Process each digit position (1s, 10s, 100s, ...)
    exp = 1
    while max_num // exp > 0:
        result = counting_sort_by_digit(result, exp)
        exp *= 10
    
    return result

In [ ]:
def radix_sort_verbose(arr: List[int]) -> List[int]:
    """
    Radix Sort with step-by-step visualization.
    
    Args:
        arr: List of non-negative integers to sort
    
    Returns:
        Sorted list
    """
    if not arr:
        return []
    
    print(f"Input array: {arr}")
    print("=" * 70)
    
    max_num = max(arr)
    num_digits = len(str(max_num))
    print(f"Maximum value: {max_num} ({num_digits} digits)")
    print(f"Number of passes required: {num_digits}")
    
    result = arr.copy()
    exp = 1
    pass_num = 1
    
    while max_num // exp > 0:
        print(f"\n{'='*70}")
        print(f"Pass {pass_num}: Sort by {'1s' if exp == 1 else f'{exp}s'} digit")
        print(f"{'='*70}")
        
        # Show digits being examined
        digits = [(num // exp) % 10 for num in result]
        print(f"\nCurrent array: {result}")
        print(f"Digits at position {exp}: {digits}")
        
        # Show bucket distribution
        buckets = {i: [] for i in range(10)}
        for num in result:
            digit = (num // exp) % 10
            buckets[digit].append(num)
        
        print(f"\nBucket distribution:")
        for digit, nums in buckets.items():
            if nums:
                print(f"  Bucket {digit}: {nums}")
        
        # Perform the sort
        result = counting_sort_by_digit(result, exp)
        print(f"\nAfter pass {pass_num}: {result}")
        
        exp *= 10
        pass_num += 1
    
    print(f"\n{'='*70}")
    print(f"FINAL SORTED ARRAY: {result}")
    print(f"{'='*70}")
    
    return result

In [ ]:
# Example: Step-by-step execution
print("RADIX SORT DEMONSTRATION")
print("=" * 70)

test_array = [170, 45, 75, 90, 802, 24, 2, 66]
result = radix_sort_verbose(test_array)

---

<a id="bucket-sort"></a>
## 3. Bucket Sort

### Theory

**Bucket Sort** distributes elements into "buckets" based on their values, sorts each bucket individually, then concatenates the buckets.

#### Key Insight
If elements are **uniformly distributed**, each bucket will have approximately the same number of elements, making the per-bucket sort very fast.

#### Assumptions & Limitations
- Works best with **uniformly distributed** data
- Input is typically assumed to be in range [0, 1) or can be normalized
- Performance degrades to O(n²) if all elements fall into one bucket
- Requires knowledge of the input distribution

#### When to Use
- Data is **uniformly distributed** across a known range
- Sorting **floating-point numbers** in [0, 1)
- When you can estimate the distribution of input

### ASCII Art Visualization

```
Input: [0.78, 0.17, 0.39, 0.26, 0.72, 0.94, 0.21, 0.12, 0.23, 0.68]
Number of buckets: 10 (one per 0.1 range)

═══════════════════════════════════════════════════════════════════════
Step 1: Distribute elements into buckets
═══════════════════════════════════════════════════════════════════════

Bucket   Range         Elements
──────────────────────────────────────────
  0    [0.0, 0.1)     []
  1    [0.1, 0.2)     [0.17, 0.12]
  2    [0.2, 0.3)     [0.26, 0.21, 0.23]
  3    [0.3, 0.4)     [0.39]
  4    [0.4, 0.5)     []
  5    [0.5, 0.6)     []
  6    [0.6, 0.7)     [0.68]
  7    [0.7, 0.8)     [0.78, 0.72]
  8    [0.8, 0.9)     []
  9    [0.9, 1.0)     [0.94]

═══════════════════════════════════════════════════════════════════════
Step 2: Sort each bucket (using insertion sort or any comparison sort)
═══════════════════════════════════════════════════════════════════════

Bucket 1: [0.17, 0.12] → [0.12, 0.17]
Bucket 2: [0.26, 0.21, 0.23] → [0.21, 0.23, 0.26]
Bucket 7: [0.78, 0.72] → [0.72, 0.78]

═══════════════════════════════════════════════════════════════════════
Step 3: Concatenate all buckets
═══════════════════════════════════════════════════════════════════════

Result: [] + [0.12, 0.17] + [0.21, 0.23, 0.26] + [0.39] + [] + [] + 
        [0.68] + [0.72, 0.78] + [] + [0.94]

Final: [0.12, 0.17, 0.21, 0.23, 0.26, 0.39, 0.68, 0.72, 0.78, 0.94] ✓
═══════════════════════════════════════════════════════════════════════
```

In [ ]:
def insertion_sort(arr: List[float]) -> List[float]:
    """
    Simple insertion sort for sorting individual buckets.
    
    Args:
        arr: List to sort (modified in place)
    
    Returns:
        The sorted list
    
    Note:
        Efficient for small arrays, which is typical for individual buckets.
    """
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr


def bucket_sort(arr: List[float], num_buckets: Optional[int] = None) -> List[float]:
    """
    Sort an array of floats in range [0, 1) using Bucket Sort.
    
    Distributes elements into buckets based on value, sorts each bucket,
    then concatenates the results.
    
    Args:
        arr: List of floats in range [0, 1) to sort
        num_buckets: Number of buckets (defaults to len(arr))
    
    Returns:
        New sorted list
    
    Time Complexity:
        - Average: O(n + k) where k = number of buckets (with uniform distribution)
        - Worst: O(n²) if all elements fall into one bucket
    
    Space Complexity: O(n) for the buckets
    
    Example:
        >>> bucket_sort([0.78, 0.17, 0.39, 0.26, 0.72, 0.94])
        [0.17, 0.26, 0.39, 0.72, 0.78, 0.94]
    """
    if not arr:
        return []
    
    n = len(arr)
    num_buckets = num_buckets or n
    
    # Create empty buckets
    buckets: List[List[float]] = [[] for _ in range(num_buckets)]
    
    # Distribute elements into buckets
    for num in arr:
        # Map value to bucket index
        # For values in [0, 1), bucket_index = floor(num * num_buckets)
        bucket_idx = min(int(num * num_buckets), num_buckets - 1)
        buckets[bucket_idx].append(num)
    
    # Sort each bucket and concatenate
    result = []
    for bucket in buckets:
        insertion_sort(bucket)  # O(k²) per bucket, but k is small
        result.extend(bucket)
    
    return result

In [ ]:
def bucket_sort_general(arr: List[float], num_buckets: Optional[int] = None) -> List[float]:
    """
    Generalized Bucket Sort that works with any range of values.
    
    Automatically normalizes input to [0, 1) range before sorting.
    
    Args:
        arr: List of floats or integers to sort
        num_buckets: Number of buckets (defaults to len(arr))
    
    Returns:
        New sorted list
    """
    if not arr:
        return []
    
    n = len(arr)
    num_buckets = num_buckets or n
    
    min_val = min(arr)
    max_val = max(arr)
    
    # Handle case where all elements are the same
    if min_val == max_val:
        return arr.copy()
    
    # Calculate range for normalization
    range_val = max_val - min_val
    
    # Create buckets
    buckets: List[List[float]] = [[] for _ in range(num_buckets)]
    
    # Distribute elements into buckets
    for num in arr:
        # Normalize to [0, 1) and find bucket
        normalized = (num - min_val) / range_val
        bucket_idx = min(int(normalized * num_buckets), num_buckets - 1)
        buckets[bucket_idx].append(num)
    
    # Sort each bucket and concatenate
    result = []
    for bucket in buckets:
        insertion_sort(bucket)
        result.extend(bucket)
    
    return result

In [ ]:
def bucket_sort_verbose(arr: List[float], num_buckets: int = 10) -> List[float]:
    """
    Bucket Sort with step-by-step visualization.
    
    Args:
        arr: List of floats in range [0, 1) to sort
        num_buckets: Number of buckets to use
    
    Returns:
        Sorted list
    """
    if not arr:
        return []
    
    print(f"Input array: {arr}")
    print(f"Number of buckets: {num_buckets}")
    print("=" * 70)
    
    # Create buckets
    buckets: List[List[float]] = [[] for _ in range(num_buckets)]
    
    # Distribute elements
    print("\nStep 1: Distribute elements into buckets")
    print("-" * 70)
    
    for num in arr:
        bucket_idx = min(int(num * num_buckets), num_buckets - 1)
        buckets[bucket_idx].append(num)
        print(f"  {num:.2f} → Bucket {bucket_idx} (range [{bucket_idx/num_buckets:.1f}, {(bucket_idx+1)/num_buckets:.1f}))")
    
    print("\nBucket contents:")
    for i, bucket in enumerate(buckets):
        range_start = i / num_buckets
        range_end = (i + 1) / num_buckets
        status = bucket if bucket else "empty"
        print(f"  Bucket {i} [{range_start:.1f}, {range_end:.1f}): {status}")
    
    # Sort each bucket
    print("\nStep 2: Sort each non-empty bucket")
    print("-" * 70)
    
    for i, bucket in enumerate(buckets):
        if len(bucket) > 1:
            before = bucket.copy()
            insertion_sort(bucket)
            print(f"  Bucket {i}: {before} → {bucket}")
    
    # Concatenate
    print("\nStep 3: Concatenate all buckets")
    print("-" * 70)
    
    result = []
    for bucket in buckets:
        result.extend(bucket)
    
    print(f"  Result: {result}")
    
    print("\n" + "=" * 70)
    print(f"FINAL SORTED ARRAY: {result}")
    print("=" * 70)
    
    return result

In [ ]:
# Example: Step-by-step execution
print("BUCKET SORT DEMONSTRATION")
print("=" * 70)

test_array = [0.78, 0.17, 0.39, 0.26, 0.72, 0.94, 0.21, 0.12, 0.23, 0.68]
result = bucket_sort_verbose(test_array)

---

<a id="complexity-comparison"></a>
## Complexity Comparison

### Summary Table

| Algorithm | Time (Best) | Time (Avg) | Time (Worst) | Space | Stable? | Constraints |
|-----------|-------------|------------|--------------|-------|---------|-------------|
| **Counting Sort** | O(n+k) | O(n+k) | O(n+k) | O(k) | Yes | Integer keys in range [0, k] |
| **Radix Sort** | O(d(n+k)) | O(d(n+k)) | O(d(n+k)) | O(n+k) | Yes | Fixed-length keys, d digits, base k |
| **Bucket Sort** | O(n+k) | O(n+k) | O(n²) | O(n) | Yes* | Uniform distribution assumed |

*Bucket Sort is stable if the underlying sorting algorithm for buckets is stable.

### Comparison with Comparison-Based Sorts

| Algorithm | Time (Best) | Time (Avg) | Time (Worst) | Space | Stable? |
|-----------|-------------|------------|--------------|-------|---------|
| Quick Sort | O(n log n) | O(n log n) | O(n²) | O(log n) | No |
| Merge Sort | O(n log n) | O(n log n) | O(n log n) | O(n) | Yes |
| Heap Sort | O(n log n) | O(n log n) | O(n log n) | O(1) | No |
| **Counting Sort** | O(n+k) | O(n+k) | O(n+k) | O(k) | Yes |
| **Radix Sort** | O(dn) | O(dn) | O(dn) | O(n+k) | Yes |
| **Bucket Sort** | O(n) | O(n) | O(n²) | O(n) | Yes* |

In [ ]:
import time
import random


def benchmark_sorts(n: int, max_val: int, num_trials: int = 3) -> dict:
    """
    Compare performance of linear sorts vs Python's built-in sort.
    
    Args:
        n: Number of elements
        max_val: Maximum value (for counting sort range)
        num_trials: Number of trials to average
    
    Returns:
        Dictionary with timing results
    """
    results = {}
    
    # Generate test data
    test_data = [random.randint(0, max_val) for _ in range(n)]
    
    # Test Python's built-in sort (Timsort)
    times = []
    for _ in range(num_trials):
        arr = test_data.copy()
        start = time.perf_counter()
        sorted(arr)
        times.append(time.perf_counter() - start)
    results['Python sorted()'] = sum(times) / len(times)
    
    # Test Counting Sort
    times = []
    for _ in range(num_trials):
        arr = test_data.copy()
        start = time.perf_counter()
        counting_sort(arr, max_val)
        times.append(time.perf_counter() - start)
    results['Counting Sort'] = sum(times) / len(times)
    
    # Test Radix Sort
    times = []
    for _ in range(num_trials):
        arr = test_data.copy()
        start = time.perf_counter()
        radix_sort(arr)
        times.append(time.perf_counter() - start)
    results['Radix Sort'] = sum(times) / len(times)
    
    # Test Bucket Sort (need to normalize data)
    normalized_data = [x / (max_val + 1) for x in test_data]
    times = []
    for _ in range(num_trials):
        arr = normalized_data.copy()
        start = time.perf_counter()
        bucket_sort(arr, n // 10 or 1)
        times.append(time.perf_counter() - start)
    results['Bucket Sort'] = sum(times) / len(times)
    
    return results

In [ ]:
# Performance comparison
print("PERFORMANCE COMPARISON")
print("=" * 70)

# Test case 1: Small range (ideal for counting sort)
print("\nTest 1: n=10,000 elements, range [0, 100] (small range)")
print("-" * 50)
results = benchmark_sorts(n=10000, max_val=100)
for name, time_taken in sorted(results.items(), key=lambda x: x[1]):
    print(f"  {name:20s}: {time_taken*1000:.3f} ms")

# Test case 2: Large range
print("\nTest 2: n=10,000 elements, range [0, 1,000,000] (large range)")
print("-" * 50)
results = benchmark_sorts(n=10000, max_val=1000000)
for name, time_taken in sorted(results.items(), key=lambda x: x[1]):
    print(f"  {name:20s}: {time_taken*1000:.3f} ms")

print("\n" + "=" * 70)
print("Observations:")
print("- Counting Sort excels when k (range) is small relative to n")
print("- Radix Sort is consistent regardless of range (depends on digits)")
print("- Bucket Sort depends heavily on uniform distribution")
print("- Python's Timsort is highly optimized for general cases")

In [ ]:
# When linear sorts shine: sorting by age
print("\nPRACTICAL EXAMPLE: Sorting People by Age")
print("=" * 70)

# Simulate sorting 1 million people by age (0-120)
n = 1_000_000
ages = [random.randint(0, 120) for _ in range(n)]

# Using Python's built-in sort
start = time.perf_counter()
sorted_ages_builtin = sorted(ages)
builtin_time = time.perf_counter() - start

# Using Counting Sort
start = time.perf_counter()
sorted_ages_counting = counting_sort(ages, 120)
counting_time = time.perf_counter() - start

print(f"Sorting {n:,} people by age (range 0-120):")
print(f"  Python sorted(): {builtin_time*1000:.1f} ms")
print(f"  Counting Sort:   {counting_time*1000:.1f} ms")
print(f"  Speedup:         {builtin_time/counting_time:.1f}x faster")

# Verify correctness
assert sorted_ages_builtin == sorted_ages_counting, "Results don't match!"
print("\n✓ Both methods produce identical results")

---

<a id="key-takeaways"></a>
## Key Takeaways

### When to Use Linear Sorts vs Comparison Sorts

| Use Linear Sorts When... | Use Comparison Sorts When... |
|--------------------------|------------------------------|
| Input is integers with small range | Input type is unknown/complex |
| Data is uniformly distributed | No assumptions about distribution |
| Memory is available for auxiliary structures | Memory is constrained |
| Stability is required | In-place sorting is needed |
| Sorting by fixed-length keys/strings | Keys have variable length |

### Decision Guide

```
┌─────────────────────────────────────────────────────────────┐
│                  WHICH SORT TO USE?                         │
└─────────────────────────────────────────────────────────────┘
                           │
                           ▼
              ┌────────────────────────┐
              │ Are elements integers? │
              └────────────────────────┘
                    │           │
                   YES          NO
                    │           │
                    ▼           ▼
         ┌──────────────┐  ┌──────────────────┐
         │ Is range     │  │ Are they floats  │
         │ k ≤ O(n)?    │  │ in [0,1)?        │
         └──────────────┘  └──────────────────┘
              │     │           │      │
             YES    NO         YES     NO
              │     │           │      │
              ▼     ▼           ▼      ▼
         ┌────────┐ │     ┌─────────┐  │
         │COUNTING│ │     │ BUCKET  │  │
         │ SORT   │ │     │  SORT   │  │
         └────────┘ │     └─────────┘  │
                    │                  │
                    ▼                  ▼
         ┌──────────────────────────────────┐
         │ Do elements have fixed #digits?  │
         └──────────────────────────────────┘
                    │           │
                   YES          NO
                    │           │
                    ▼           ▼
              ┌─────────┐  ┌──────────────┐
              │  RADIX  │  │ COMPARISON   │
              │  SORT   │  │ SORT (Quick/ │
              └─────────┘  │ Merge/Heap)  │
                           └──────────────┘
```

### Trade-offs Summary

| Factor | Counting Sort | Radix Sort | Bucket Sort |
|--------|---------------|------------|-------------|
| **Space** | O(k) - can be large | O(n+k) - moderate | O(n) - moderate |
| **Predictability** | Very predictable | Very predictable | Depends on distribution |
| **Input constraints** | Small integer range | Fixed-length keys | Uniform distribution |
| **Best use case** | Ages, grades, counts | SSNs, dates, IPs | Random floats |

### Final Notes

1. **Linear sorts aren't always faster** - The constants matter, and comparison sorts are highly optimized.

2. **Know your data** - Linear sorts require assumptions. If violated, performance can degrade significantly.

3. **Stability matters** - All three linear sorts can be stable, which is valuable for multi-key sorting.

4. **Space vs Time** - Linear sorts trade space for time. If memory is constrained, comparison sorts may be better.

5. **Radix Sort is versatile** - Works on integers, strings, and any data with lexicographic ordering.

In [ ]:
# Final verification: All implementations work correctly
print("FINAL VERIFICATION")
print("=" * 70)

# Test Counting Sort
test1 = [4, 2, 2, 8, 3, 3, 1]
assert counting_sort(test1) == sorted(test1), "Counting Sort failed!"
print("✓ Counting Sort: PASSED")

# Test Radix Sort
test2 = [170, 45, 75, 90, 802, 24, 2, 66]
assert radix_sort(test2) == sorted(test2), "Radix Sort failed!"
print("✓ Radix Sort: PASSED")

# Test Bucket Sort
test3 = [0.78, 0.17, 0.39, 0.26, 0.72, 0.94, 0.21, 0.12, 0.23, 0.68]
assert bucket_sort(test3) == sorted(test3), "Bucket Sort failed!"
print("✓ Bucket Sort: PASSED")

# Test General Bucket Sort
test4 = [78, 17, 39, 26, 72, 94, 21, 12, 23, 68]
assert bucket_sort_general(test4) == sorted(test4), "General Bucket Sort failed!"
print("✓ General Bucket Sort: PASSED")

# Edge cases
assert counting_sort([]) == [], "Empty array failed!"
assert counting_sort([5]) == [5], "Single element failed!"
assert counting_sort([1, 1, 1]) == [1, 1, 1], "All same elements failed!"
print("✓ Edge cases: PASSED")

print("\n" + "=" * 70)
print("All tests passed! Implementations are correct.")

---

## Exercises

### Exercise 1: Counting Sort on DNA Nucleotides (1 star)

Use counting sort to rearrange a DNA sequence string so that all A's appear first, then all C's, then G's, then T's. Count the total number of comparison-free assignment operations performed (i.e., the number of times you write a character into the output array).

**Biological context**: Computing nucleotide composition and generating a sorted nucleotide string is a preprocessing step in k-mer analysis and suffix array construction. Counting sort is ideal here because the alphabet is fixed at size 4, giving true O(n) performance regardless of sequence length.

In [ ]:
def counting_sort_dna(sequence: str) -> tuple[str, int]:
    """
    Sort a DNA string so bases appear in order: A, C, G, T.
    Return (sorted_sequence, operation_count) where operation_count
    is the number of character writes into the output array.
    """
    # YOUR CODE HERE
    pass


# Test
# dna = 'TGCAATCGGCTATCGAATCGATCGATCGATCGATCGGGCCTA'
# sorted_dna, ops = counting_sort_dna(dna)
# print(f'Input:    {dna}')
# print(f'Sorted:   {sorted_dna}')
# print(f'Length:   {len(dna)}')
# print(f'Writes:   {ops}  (should equal {len(dna)})')
# counts = {b: sorted_dna.count(b) for b in 'ACGT'}
# print(f'Composition: {counts}')

### Exercise 2: Radix Sort of k-mers vs Built-in sorted (2 stars)

Extract all 3-mers from a DNA sequence and sort them lexicographically using radix sort (process characters right-to-left, one position at a time using counting sort as the stable subroutine). Compare the result and runtime against Python's built-in `sorted()`.

**Biological context**: k-mer sorting is the first step in constructing de Bruijn graphs used in genome assembly (e.g., Velvet, SPAdes). Radix sort is attractive here because the alphabet is small (size 4) and all k-mers have the same fixed length, which are exactly the conditions where radix sort outperforms comparison-based sorting.

In [ ]:
import time


def extract_kmers(sequence: str, k: int) -> list[str]:
    """Return all k-mers (substrings of length k) from sequence."""
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]


def radix_sort_kmers(kmers: list[str]) -> list[str]:
    """
    Sort fixed-length DNA k-mers lexicographically using radix sort.
    Process character positions from rightmost to leftmost.
    Use counting sort with alphabet 'ACGT' as the stable subroutine.
    """
    # YOUR CODE HERE
    pass


# Test and benchmark
# import random
# dna = ''.join(random.choices('ACGT', k=10_000))
# kmers = extract_kmers(dna, k=3)
# print(f'Total 3-mers: {len(kmers)}')
#
# t0 = time.perf_counter()
# radix_result = radix_sort_kmers(kmers[:])
# t1 = time.perf_counter()
# builtin_result = sorted(kmers)
# t2 = time.perf_counter()
#
# print(f'Radix sort:      {(t1 - t0) * 1000:.2f} ms')
# print(f'Built-in sorted: {(t2 - t1) * 1000:.2f} ms')
# print(f'Results match:   {radix_result == builtin_result}')
# print(f'First 10 k-mers: {radix_result[:10]}')

---

[← Previous: Comparison-Based Sorting Algorithms](../../Tier_4_Algorithms_and_Data_Structures/02_Sorting_Algorithms/01_comparison_sorts.ipynb) | [Next: Linear and Binary Search Algorithms →](../../Tier_4_Algorithms_and_Data_Structures/03_Searching_Algorithms/01_linear_binary_search.ipynb)